In [1]:
#imports das bibliotecas e carregamento ods dados

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from src.preparacao import carregar_base, separar_x_y

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, precision_recall_curve)

base = carregar_base()
print(base[["y_abastecimento", "y_setup", "y_parada_longa"]].mean().round(3))

y_abastecimento    0.723
y_setup            0.399
y_parada_longa     0.093
dtype: float64


In [2]:
#funções auxiliares para facilitar a execução do notebook

def separar_tempo(X, y, frac_dev=0.8):
    """80% mais antigos = desenvolvimento; 20% mais recentes = teste final."""
    corte = int(len(X) * frac_dev)
    return X.iloc[:corte], X.iloc[corte:], y.iloc[:corte], y.iloc[corte:]


def grades():
    """Modelos e valores a testar (mesmos da Fase 4 + class_weight)."""
    return {
        "Regressão Logística": (
            make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)),
            {"logisticregression__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10],
             "logisticregression__class_weight": [None, "balanced"]}),
        "Árvore de Decisão": (
            make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(random_state=42)),
            {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10],
             "decisiontreeclassifier__min_samples_leaf": [5, 10, 20, 50, 100],
             "decisiontreeclassifier__class_weight": [None, "balanced"]}),
        "Random Forest": (
            make_pipeline(SimpleImputer(strategy="median"),
                          RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)),
            {"randomforestclassifier__max_depth": [None, 6, 10, 15],
             "randomforestclassifier__min_samples_leaf": [1, 3, 5, 10, 20],
             "randomforestclassifier__max_features": ["sqrt", 0.3, 0.5],
             "randomforestclassifier__class_weight": [None, "balanced_subsample"]}),
        "Gradient Boosting": (
            HistGradientBoostingClassifier(random_state=42),
            {"learning_rate": [0.02, 0.05, 0.1],
             "max_iter": [100, 200, 400],
             "max_depth": [None, 3, 5],
             "min_samples_leaf": [10, 20, 50],
             "l2_regularization": [0, 0.1, 1.0],
             "class_weight": [None, "balanced"]}),
    }


def buscar_melhores(X_dev, y_dev):
    """Busca de hiperparâmetros com validação temporal; devolve o melhor de cada algoritmo."""
    melhores = {}
    for nome, (modelo, grade) in grades().items():
        busca = RandomizedSearchCV(modelo, grade, n_iter=20, scoring="roc_auc",
                                   cv=TimeSeriesSplit(n_splits=5), random_state=42, n_jobs=-1)
        busca.fit(X_dev, y_dev)
        melhores[nome] = busca.best_estimator_
        print(f"{nome}: AUC na validação = {busca.best_score_:.3f}")
        print("   parâmetros:", busca.best_params_)
    return melhores


def avaliar(modelos, X_test, y_test, limiar=0.5):
    """Mede todos os modelos no teste final."""
    linhas = []
    for nome, modelo in modelos.items():
        prob = modelo.predict_proba(X_test)[:, 1]
        pred = (prob >= limiar).astype(int)
        linhas.append({"modelo": nome,
                       "acurácia": accuracy_score(y_test, pred),
                       "precisão": precision_score(y_test, pred, zero_division=0),
                       "recall":   recall_score(y_test, pred),
                       "F1":       f1_score(y_test, pred, zero_division=0),
                       "AUC":      roc_auc_score(y_test, prob),
                       "AP":       average_precision_score(y_test, prob)})
    return pd.DataFrame(linhas)


def importancia(modelo, X_test, y_test, n=10):
    """Importância por permutação: quanto a AUC cai ao embaralhar cada variável."""
    imp = permutation_importance(modelo, X_test, y_test, scoring="roc_auc",
                                 n_repeats=10, random_state=42, n_jobs=-1)
    return (pd.DataFrame({"variável": X_test.columns,
                          "queda_AUC": imp.importances_mean,
                          "desvio": imp.importances_std})
            .sort_values("queda_AUC", ascending=False).head(n).round(4))

In [ ]:
#separar e ajustar setup

X, y = separar_x_y(base, alvo="y_setup")
X_dev, X_test, y_dev, y_test = separar_tempo(X, y)
print(f"Taxa de setup – desenvolvimento: {y_dev.mean():.3f} | teste: {y_test.mean():.3f}")

melhores_setup = buscar_melhores(X_dev, y_dev)

Taxa de setup – desenvolvimento: 0.381 | teste: 0.467


c:\Users\davif\Documents\FHO\8º Período\Inteligência Artificial I\IA Preditiva - MC 68 e MC 63\IA-Preditiva\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 14 is smaller than n_iter=20. Running 14 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Regressão Logística: AUC na validação = 0.605
   parâmetros: {'logisticregression__class_weight': None, 'logisticregression__C': 0.01}
Árvore de Decisão: AUC na validação = 0.803
   parâmetros: {'decisiontreeclassifier__min_samples_leaf': 10, 'decisiontreeclassifier__max_depth': 6, 'decisiontreeclassifier__class_weight': 'balanced'}
Random Forest: AUC na validação = 0.818
   parâmetros: {'randomforestclassifier__min_samples_leaf': 3, 'randomforestclassifier__max_features': 0.5, 'randomforestclassifier__max_depth': None, 'randomforestclassifier__class_weight': None}
Gradient Boosting: AUC na validação = 0.824
   parâmetros: {'min_samples_leaf': 20, 'max_iter': 100, 'max_depth': 5, 'learning_rate': 0.02, 'l2_regularization': 0, 'class_weight': 'balanced'}


In [7]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)
res_setup = avaliar({"Baseline": baseline, **melhores_setup}, X_test, y_test)
res_setup.round(3)

,modelo,acurácia,precisão,recall,F1,AUC,AP
0,Baseline,0.533,0.000,0.000,0.000,0.500,0.467
1,Regressão Logística,0.589,0.685,0.224,0.338,0.644,0.613
2,Árvore de Decisão,0.761,0.751,0.731,0.741,0.793,0.770
3,Random Forest,0.772,0.821,0.654,0.728,0.861,0.813
4,Gradient Boosting,0.785,0.792,0.734,0.762,0.867,0.824


In [ ]:
#desvio
importancia(melhores_setup["Gradient Boosting"], X_test, y_test)

,variável,queda_AUC,desvio
14,hora_ini,0.2752,0.0183
23,mc_68,0.1341,0.0088
6,troca_terminal_b,0.0193,0.0023
9,troca_ma_b,0.0180,0.0024
4,troca_condutor,0.0114,0.0023
18,n_ocorrencias_previas_sub,0.0110,0.0044
5,troca_terminal_a,0.0098,0.0029
8,troca_ma_a,0.0096,0.0025
12,ocioso_antes_min,0.0063,0.0024
20,razao_media_hist_sub,0.0035,0.0014


hora de inicio e a estação de trabalho foram as variáveis com mais queda na AUC, o contrário do esperado. Vale a investigação

In [ ]:
#investigando as quedas na AUC
# Teve alguma troca de material em relação à ordem anterior?
trocas = ["troca_condutor", "troca_terminal_a", "troca_terminal_b", "troca_ma_a", "troca_ma_b"]
base["teve_troca"] = (base[trocas].fillna(0).sum(axis=1) > 0).astype(int)

# Minuto do dia em que a sessão começou (ex.: 05:45 = 5*60 + 45 = 345)
minuto = base["ini"].dt.hour * 60 + base["ini"].dt.minute

base["turno"] = np.select(
    [(minuto >= 345) & (minuto < 720),       # 05:45 – 12:00
     (minuto >= 720) & (minuto < 1095),      # 12:00 – 18:15
     (minuto >= 1095) | (minuto < 9)],       # 18:15 – 00:09 (atravessa a meia-noite)
    ["T1", "T2", "T3"],
    default="T4")                            # 00:09 – 05:45

print(base["turno"].value_counts().sort_index())

# % de sessões com setup apontado × % com troca real de material, por máquina e turno
(base.groupby(["mc", "turno"])[["y_setup", "teve_troca"]].mean().mul(100).round(0).unstack(0))

As trocas reais de material são estáveis entre os turnos (62–68% na MC 63; 41–50% na MC 68), mas o apontamento de setup varia de 8% a 81% conforme turno e máquina. O modelo de setup aprende o padrão de apontamento, não o evento físico.

questionamento muito válido sobre os 8% de setup na 63 no terceiro turno, há um viés sobre operadores selecionarem somente os "lotes gordos" e deixarem a "picadeira" para outros turnos, analisar a possibilidade de ser isso, tendo em vista que a programação é liberada no final do 2t e inicio do 3t.(mc 68 é a mais rapida do parque, o que justifica a troca diminuir no t1 e t2 devido a estar adiantada na programação) Isso com certeza vai mudar após travarmos a programação de acordo com o melhor setup.


In [9]:
#separar e ajustar parada longa

X, y = separar_x_y(base, alvo="y_parada_longa")
X_dev, X_test, y_dev, y_test = separar_tempo(X, y)
print(f"Taxa de parada longa – desenvolvimento: {y_dev.mean():.3f} | teste: {y_test.mean():.3f}")

melhores_longa = buscar_melhores(X_dev, y_dev)

Taxa de parada longa – desenvolvimento: 0.084 | teste: 0.131


c:\Users\davif\Documents\FHO\8º Período\Inteligência Artificial I\IA Preditiva - MC 68 e MC 63\IA-Preditiva\.venv\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 14 is smaller than n_iter=20. Running 14 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Regressão Logística: AUC na validação = 0.650
   parâmetros: {'logisticregression__class_weight': None, 'logisticregression__C': 0.01}
Árvore de Decisão: AUC na validação = 0.638
   parâmetros: {'decisiontreeclassifier__min_samples_leaf': 100, 'decisiontreeclassifier__max_depth': 3, 'decisiontreeclassifier__class_weight': None}
Random Forest: AUC na validação = 0.660
   parâmetros: {'randomforestclassifier__min_samples_leaf': 10, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__max_depth': 6, 'randomforestclassifier__class_weight': None}
Gradient Boosting: AUC na validação = 0.656
   parâmetros: {'min_samples_leaf': 10, 'max_iter': 100, 'max_depth': 3, 'learning_rate': 0.02, 'l2_regularization': 0.1, 'class_weight': None}


In [10]:
#teste com o corte padrão de 0.5

baseline = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)
res_longa_05 = avaliar({"Baseline": baseline, **melhores_longa}, X_test, y_test, limiar=0.5)
res_longa_05.round(3)

,modelo,acurácia,precisão,recall,F1,AUC,AP
0,Baseline,0.869,0.0,0.0,0.0,0.500,0.131
1,Regressão Logística,0.868,0.0,0.0,0.0,0.732,0.291
2,Árvore de Decisão,0.869,0.0,0.0,0.0,0.638,0.207
3,Random Forest,0.869,0.0,0.0,0.0,0.746,0.309
4,Gradient Boosting,0.869,0.0,0.0,0.0,0.752,0.312


In [11]:
#avaliar diferentes pontos de corte
def escolher_limiar(modelo, X_dev, y_dev):
    """Treina nos 75% iniciais do desenvolvimento e escolhe o corte que maximiza o F1 nos 25% finais."""
    c = int(len(X_dev) * 0.75)
    m = clone(modelo).fit(X_dev.iloc[:c], y_dev.iloc[:c])
    prob = m.predict_proba(X_dev.iloc[c:])[:, 1]
    prec, rec, lim = precision_recall_curve(y_dev.iloc[c:], prob)   # precisão e recall para cada corte possível
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return lim[np.argmax(f1[:-1])]                                 # corte com o maior F1

limiares = {nome: escolher_limiar(m, X_dev, y_dev) for nome, m in melhores_longa.items()}
pd.Series(limiares).round(3)

Regressão Logística    0.155
Árvore de Decisão      0.203
Random Forest          0.114
Gradient Boosting      0.123
dtype: float64

In [12]:
#teste com ponto de corte escolhido para cada modelo
linhas = []
for nome, modelo in melhores_longa.items():
    linhas.append(avaliar({nome: modelo}, X_test, y_test, limiar=limiares[nome])
                  .assign(limiar=limiares[nome]))
res_longa = pd.concat(linhas, ignore_index=True)

print(f"Paradas longas reais no teste: {y_test.sum()} de {len(y_test)} sessões")
res_longa.round(3)

Paradas longas reais no teste: 106 de 811 sessões


,modelo,acurácia,precisão,recall,F1,AUC,AP,limiar
0,Regressão Logística,0.837,0.319,0.217,0.258,0.732,0.291,0.155
1,Árvore de Decisão,0.821,0.311,0.302,0.306,0.638,0.207,0.203
2,Random Forest,0.758,0.282,0.547,0.372,0.746,0.309,0.114
3,Gradient Boosting,0.800,0.306,0.415,0.352,0.752,0.312,0.123


In [13]:
importancia(melhores_longa["Gradient Boosting"], X_test, y_test)

,variável,queda_AUC,desvio
12,ocioso_antes_min,0.0459,0.0111
2,tempo_prog_sessao_min,0.0263,0.0183
14,hora_ini,0.0176,0.0039
4,troca_condutor,0.0176,0.0063
22,maquina_diferente_programada,0.0144,0.0047
5,troca_terminal_a,0.0118,0.0029
0,comprimento,0.0092,0.0050
7,troca_cor,0.0087,0.0040
20,razao_media_hist_sub,0.0080,0.0047
19,dur_media_hist_sub,0.0071,0.0064
